<a href="https://colab.research.google.com/github/duruamobi/AAI2026/blob/main/Multi_Agent_System.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
"""
Multi-Agent Agentic AI System for IT Support

This prototype demonstrates:
- Product ownership thinking
- Multi-agent orchestration
- RAG-style knowledge retrieval
- Workflow automation
- MCP-style tool integration
- Escalation handling
- Testing and validation metrics
"""

from dataclasses import dataclass, field
from typing import Dict, List, Optional
import time
import uuid


# -----------------------------
# Product Owner Configuration
# -----------------------------

PRODUCT_SCOPE = {
    "problem": "Employees lose productivity waiting for IT help on common issues.",
    "target_users": ["Employees", "IT Support Analysts", "IT Managers"],
    "mvp_use_cases": [
        "password_reset",
        "vpn_issue",
        "software_access",
        "ticket_triage",
        "knowledge_base_question",
    ],
    "success_metrics": {
        "avg_response_time_seconds": 10,
        "first_contact_resolution_target": 0.70,
        "escalation_accuracy_target": 0.90,
        "user_satisfaction_target": 4.0,
    },
}


# -----------------------------
# Shared State
# -----------------------------

@dataclass
class SupportState:
    user_message: str
    user_id: str = "employee_001"
    category: Optional[str] = None
    urgency: str = "normal"
    requires_escalation: bool = False
    retrieved_docs: List[str] = field(default_factory=list)
    workflow_result: Optional[str] = None
    final_answer: Optional[str] = None
    ticket_id: Optional[str] = None
    start_time: float = field(default_factory=time.time)


# -----------------------------
# RAG Knowledge Base
# -----------------------------

class KnowledgeBase:
    def __init__(self):
        self.docs = {
            "password_reset": """
            Password Reset Policy:
            Users must verify identity before resetting passwords.
            Use the self-service portal first. If MFA is unavailable, escalate to IT.
            """,
            "vpn_issue": """
            VPN Troubleshooting:
            1. Confirm internet connection.
            2. Restart VPN client.
            3. Check MFA prompt.
            4. Try a different network.
            5. If error persists, create a ticket with device OS and error code.
            """,
            "software_access": """
            Software Access:
            Users need manager approval for paid software.
            Standard tools can be requested through the IT service portal.
            Admin permissions require escalation.
            """,
            "device_issue": """
            Device Troubleshooting:
            Restart the device, check updates, verify disk space, and collect logs.
            Hardware failure should be escalated to desktop support.
            """,
        }

    def retrieve(self, query: str, category: str) -> List[str]:
        results = []

        if category in self.docs:
            results.append(self.docs[category])

        for key, doc in self.docs.items():
            if key.replace("_", " ") in query.lower() and doc not in results:
                results.append(doc)

        return results[:2]


# -----------------------------
# MCP-Style Tool Layer
# -----------------------------

class MCPToolRouter:
    """
    Simulates Model Context Protocol style tool access.
    In a real implementation, these could connect to Jira, GitHub,
    Slack, Okta, ServiceNow, or endpoint management tools.
    """

    def check_service_status(self, service_name: str) -> Dict:
        mock_status = {
            "vpn": "operational",
            "email": "operational",
            "sso": "degraded",
        }
        return {
            "service": service_name,
            "status": mock_status.get(service_name, "unknown"),
        }

    def create_ticket(self, user_id: str, category: str, summary: str, urgency: str) -> str:
        ticket_id = f"IT-{uuid.uuid4().hex[:6].upper()}"
        print("\n[MCP TOOL] Ticket created")
        print(f"Ticket ID: {ticket_id}")
        print(f"User: {user_id}")
        print(f"Category: {category}")
        print(f"Urgency: {urgency}")
        print(f"Summary: {summary}\n")
        return ticket_id

    def reset_password_workflow(self, user_id: str) -> str:
        return (
            f"Identity verification required for {user_id}. "
            "Password reset link sent through the self-service portal."
        )


# -----------------------------
# Agents
# -----------------------------

class IntakeAgent:
    def run(self, state: SupportState) -> SupportState:
        msg = state.user_message.lower()

        if "password" in msg or "login" in msg:
            state.category = "password_reset"
        elif "vpn" in msg:
            state.category = "vpn_issue"
        elif "software" in msg or "install" in msg or "access" in msg:
            state.category = "software_access"
        elif "laptop" in msg or "device" in msg or "computer" in msg:
            state.category = "device_issue"
        else:
            state.category = "general_it"

        if any(word in msg for word in ["urgent", "blocked", "production", "security breach"]):
            state.urgency = "high"

        return state


class SecurityAgent:
    def run(self, state: SupportState) -> SupportState:
        risky_terms = ["admin access", "delete account", "disable mfa", "security breach"]

        if any(term in state.user_message.lower() for term in risky_terms):
            state.requires_escalation = True

        return state


class KnowledgeAgent:
    def __init__(self, kb: KnowledgeBase):
        self.kb = kb

    def run(self, state: SupportState) -> SupportState:
        state.retrieved_docs = self.kb.retrieve(state.user_message, state.category)
        return state


class WorkflowAgent:
    def __init__(self, tools: MCPToolRouter):
        self.tools = tools

    def run(self, state: SupportState) -> SupportState:
        if state.requires_escalation:
            state.workflow_result = "Workflow blocked because request requires human review."
            return state

        if state.category == "password_reset":
            state.workflow_result = self.tools.reset_password_workflow(state.user_id)

        elif state.category == "vpn_issue":
            status = self.tools.check_service_status("vpn")
            state.workflow_result = f"VPN service status: {status['status']}."

        elif state.category == "software_access":
            state.workflow_result = (
                "Software request prepared. Manager approval may be required."
            )

        else:
            state.workflow_result = "No automated workflow available for this issue."

        return state


class EscalationAgent:
    def __init__(self, tools: MCPToolRouter):
        self.tools = tools

    def run(self, state: SupportState) -> SupportState:
        summary = (
            f"User issue: {state.user_message}\n"
            f"Category: {state.category}\n"
            f"Retrieved context: {state.retrieved_docs}\n"
            f"Workflow result: {state.workflow_result}"
        )

        state.ticket_id = self.tools.create_ticket(
            user_id=state.user_id,
            category=state.category or "unknown",
            summary=summary,
            urgency=state.urgency,
        )

        return state


class ResponseAgent:
    def run(self, state: SupportState) -> SupportState:
        docs = "\n".join(state.retrieved_docs) if state.retrieved_docs else "No matching knowledge article found."

        if state.ticket_id:
            state.final_answer = f"""
I created an IT ticket for this issue.

Ticket: {state.ticket_id}
Category: {state.category}
Urgency: {state.urgency}

What I found:
{docs}

Automation result:
{state.workflow_result}

A human IT analyst should review this next.
"""
        else:
            state.final_answer = f"""
I can help with that.

Category: {state.category}
Urgency: {state.urgency}

Recommended steps:
{docs}

Automation result:
{state.workflow_result}

If this does not solve the issue, I can escalate it to IT.
"""
        return state


# -----------------------------
# Orchestrator
# -----------------------------

class ITSupportOrchestrator:
    def __init__(self):
        self.kb = KnowledgeBase()
        self.tools = MCPToolRouter()

        self.intake_agent = IntakeAgent()
        self.security_agent = SecurityAgent()
        self.knowledge_agent = KnowledgeAgent(self.kb)
        self.workflow_agent = WorkflowAgent(self.tools)
        self.escalation_agent = EscalationAgent(self.tools)
        self.response_agent = ResponseAgent()

    def handle_request(self, user_message: str, user_id: str = "employee_001") -> SupportState:
        state = SupportState(user_message=user_message, user_id=user_id)

        state = self.intake_agent.run(state)
        state = self.security_agent.run(state)
        state = self.knowledge_agent.run(state)
        state = self.workflow_agent.run(state)

        if state.requires_escalation or state.category == "general_it":
            state = self.escalation_agent.run(state)

        state = self.response_agent.run(state)
        return state


# -----------------------------
# Validation & Testing
# -----------------------------

def run_validation_tests():
    test_cases = [
        {
            "input": "I forgot my password and cannot login.",
            "expected_category": "password_reset",
        },
        {
            "input": "My VPN will not connect.",
            "expected_category": "vpn_issue",
        },
        {
            "input": "I need access to install design software.",
            "expected_category": "software_access",
        },
        {
            "input": "I need admin access to disable MFA.",
            "expected_escalation": True,
        },
    ]

    app = ITSupportOrchestrator()
    correct = 0
    total_time = 0

    for test in test_cases:
        start = time.time()
        result = app.handle_request(test["input"])
        elapsed = time.time() - start
        total_time += elapsed

        category_ok = result.category == test.get("expected_category", result.category)
        escalation_ok = result.requires_escalation == test.get(
            "expected_escalation",
            result.requires_escalation,
        )

        if category_ok and escalation_ok:
            correct += 1

    accuracy = correct / len(test_cases)
    avg_time = total_time / len(test_cases)

    print("\n--- Validation Metrics ---")
    print(f"Classification / routing accuracy: {accuracy:.0%}")
    print(f"Average response time: {avg_time:.4f} seconds")
    print(f"Test cases executed: {len(test_cases)}")


# -----------------------------
# Simple UX: Command-Line Chat
# -----------------------------

def main():
    app = ITSupportOrchestrator()

    print("===================================")
    print(" IT AssistAI - Multi-Agent Support ")
    print("===================================")
    print("Type an IT issue or type 'test' to run validation.")
    print("Type 'exit' to quit.\n")

    while True:
        user_input = input("Employee: ")

        if user_input.lower() == "exit":
            print("Goodbye.")
            break

        if user_input.lower() == "test":
            run_validation_tests()
            continue

        result = app.handle_request(user_input)
        print("\nAssistant:")
        print(result.final_answer)


if __name__ == "__main__":
    main()

 IT AssistAI - Multi-Agent Support 
Type an IT issue or type 'test' to run validation.
Type 'exit' to quit.

Employee: Give me admin access

[MCP TOOL] Ticket created
Ticket ID: IT-99F465
User: employee_001
Category: software_access
Urgency: normal
Summary: User issue: Give me admin access
Category: software_access
Retrieved context: ['\n            Software Access:\n            Users need manager approval for paid software.\n            Standard tools can be requested through the IT service portal.\n            Admin permissions require escalation.\n            ']
Workflow result: Workflow blocked because request requires human review.


Assistant:

I created an IT ticket for this issue.

Ticket: IT-99F465
Category: software_access
Urgency: normal

What I found:

            Software Access:
            Users need manager approval for paid software.
            Standard tools can be requested through the IT service portal.
            Admin permissions require escalation.
            

KeyboardInterrupt: Interrupted by user